In [14]:
%%capture
!pip install unsloth
!pip install accelerate==0.33.0
!pip install --upgrade numpy
!pip install gguf
!git clone https://github.com/ggml-org/llama.cpp.git


In [1]:
# Step 1: Upload your adapter files
from google.colab import files
print("Upload adapter_config.json and adapter_model.safetensors:")
uploaded = files.upload()

# Step 2: Organize
!mkdir -p julia_microtune_adapter
!mv adapter_config.json adapter_model.safetensors julia_microtune_adapter/

# Step 3: Merge adapter into base model
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "julia_microtune_adapter",
    max_seq_length = 1024,
    dtype = None,
    load_in_4bit = True,
)
# Save merged 16-bit
model.save_pretrained_merged("julia_merged", tokenizer, save_method="merged_16bit")
print("✅ Merged!")

# Step 4: Convert to GGUF
!python llama.cpp/convert_hf_to_gguf.py julia_merged --outfile julia-scicore-q4_k_m.gguf --outtype q4_k_m
print("✅ GGUF created!")

# Step 5: Download
import os
size = os.path.getsize("julia-scicore-q4_k_m.gguf") / (1024**3)
print(f"Downloading julia-scicore-q4_k_m.gguf ({size:.2f} GB)")
files.download("julia-scicore-q4_k_m.gguf")


Upload adapter_config.json and adapter_model.safetensors:


Saving adapter_config.json to adapter_config.json
Saving adapter_model.safetensors to adapter_model.safetensors
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Qwen3 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.50G [00:00<?, ?B/s]

...

Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [00:49<00:00, 12.44s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [00:53<00:00, 13.29s/it]


Unsloth: Merge process complete. Saved to `/content/julia_merged`
✅ Merged!
usage: convert_hf_to_gguf.py [-h] [--vocab-only] [--outfile OUTFILE]
                             [--outtype {f32,f16,bf16,q8_0,tq1_0,tq2_0,auto}]
                             [--bigendian] [--use-temp-file] [--no-lazy]
                             [--model-name MODEL_NAME] [--verbose]
                             [--split-max-tensors SPLIT_MAX_TENSORS]
                             [--split-max-size SPLIT_MAX_SIZE] [--dry-run]
                             [--no-tensor-first-split] [--metadata METADATA]
                             [--print-supported-models] [--remote] [--mmproj]
                             [--mistral-format]
                             [--disable-mistral-community-chat-template]
                             [--sentence-transformers-dense-modules]
                             [--fuse-gate-up-exps]
                             [model]
convert_hf_to_gguf.py: error: argument --outtype: invalid ch

FileNotFoundError: [Errno 2] No such file or directory: 'julia-scicore-q4_k_m.gguf'

In [3]:
# Step 1: Convert to f16 GGUF first
!python llama.cpp/convert_hf_to_gguf.py julia_merged --outfile julia-scicore-f16.gguf --outtype f16

# Step 2: Build the quantizer
!cd llama.cpp && cmake -B build && cmake --build build --config Release -j --target llama-quantize

# Step 3: Quantize to Q4_K_M
!./llama.cpp/build/bin/llama-quantize julia-scicore-f16.gguf julia-scicore-q4_k_m.gguf Q4_K_M

# Step 4: Download
import os
from google.colab import files
size = os.path.getsize("julia-scicore-q4_k_m.gguf") / (1024**3)
print(f"📦 julia-scicore-q4_k_m.gguf ({size:.2f} GB)")
files.download("julia-scicore-q4_k_m.gguf")


INFO:hf-to-gguf:Loading model: julia_merged
INFO:hf-to-gguf:Model architecture: Qwen3ForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00004.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00004.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00003-of-00004.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00004-of-00004.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.bfloat16 --> F16, shape = {4096, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.bfloat16 --> F32, shape = {4096}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.bfloat16 --> F16, shape = {12288, 4096}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.bfloat16 --> F16, shape = {4096, 12288}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.bfloat16 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>